# Notebook 03 — CutMix Augmentation

**Session:** Image Augmentation  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement **CutMix** regularization: paste a randomly sampled rectangular patch from one mini-batch image onto another and blend the cross-entropy loss proportionally to the unoccluded pixel area $\lambda$.

**Learning objectives:**
- Sample Beta-distributed box ratios and clip coordinates via `rand_bbox`.
- Recompute the exact pixel area ratio $\lambda_{\text{adj}} = 1 - \frac{(x_2 - x_1)(y_2 - y_1)}{W \cdot H}$.
- Train a ResNet-18 classifier with mixed targets on portable CPU/GPU devices.

In [ ]:
from pathlib import Path

import albumentations
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

## 1  CutMix Bounding Box & Training Loop

In [ ]:
def get_accuracy(model: nn.Module, data_loader: DataLoader, device: torch.device) -> float:
    """Compute classification accuracy (%) over an entire DataLoader."""
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100.0 * (correct / total) if total > 0 else 0.0


def plot_losses(
    train_acc: list[float],
    valid_acc: list[float],
    train_loss: list[float],
    valid_loss: list[float],
) -> None:
    """Plot training and validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(train_acc, color="tab:blue", label="Train Accuracy")
    ax1.plot(valid_acc, color="tab:red", label="Validation Accuracy")
    ax1.set(title="Accuracy over Epochs", xlabel="Epoch", ylabel="Accuracy (%)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(train_loss, color="tab:blue", label="Train Loss")
    ax2.plot(valid_loss, color="tab:red", label="Validation Loss")
    ax2.set(title="Loss over Epochs", xlabel="Epoch", ylabel="Loss")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    plt.show()


def validate(
    valid_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[nn.Module, float]:
    """Run one validation epoch without gradient tracking."""
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

    epoch_loss = running_loss / max(len(valid_loader), 1)
    return model, epoch_loss


def rand_bbox(size: torch.Size, lam: float) -> tuple[int, int, int, int]:
    """Sample a random rectangular bounding box with area ratio (1 - lam)."""
    height = size[2]
    width = size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_h = int(height * cut_rat)
    cut_w = int(width * cut_rat)

    cy = int(np.random.randint(height))
    cx = int(np.random.randint(width))

    y1 = int(np.clip(cy - cut_h // 2, 0, height))
    x1 = int(np.clip(cx - cut_w // 2, 0, width))
    y2 = int(np.clip(cy + cut_h // 2, 0, height))
    x2 = int(np.clip(cx + cut_w // 2, 0, width))

    return y1, x1, y2, x2


def train_cutmix(
    train_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    cutmix_beta: float = 1.0,
    cutmix_prob: float = 1.0,
) -> tuple[nn.Module, torch.optim.Optimizer, float]:
    """Run one epoch with CutMix patch swapping and area-weighted target mixing."""
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        if np.random.rand() < cutmix_prob:
            lam = float(np.random.beta(cutmix_beta, cutmix_beta))
            rand_index = torch.randperm(images.size(0), device=device)
            target_a = labels
            target_b = labels[rand_index]

            y1, x1, y2, x2 = rand_bbox(images.size(), lam)
            images[:, :, y1:y2, x1:x2] = images[rand_index, :, y1:y2, x1:x2]
            lam = 1.0 - ((y2 - y1) * (x2 - x1) / (images.size(-2) * images.size(-1)))

            outputs = model(images)
            loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1.0 - lam)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, optimizer, running_loss / max(len(train_loader), 1)


def training_loop(
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int,
    device: torch.device,
) -> tuple[nn.Module, torch.optim.Optimizer, tuple[list[float], list[float]]]:
    """Train with CutMix and evaluate on clean validation batches."""
    train_losses: list[float] = []
    valid_losses: list[float] = []
    list_train_acc: list[float] = []
    list_val_acc: list[float] = []

    for epoch in range(epochs):
        model, optimizer, train_loss = train_cutmix(
            train_loader, model, criterion, optimizer, device
        )
        model, valid_loss = validate(valid_loader, model, criterion, device)
        train_acc = get_accuracy(model, train_loader, device=device)
        valid_acc = get_accuracy(model, valid_loader, device=device)

        print(
            f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Valid loss: {valid_loss:.4f} "
            f"| Train acc: {train_acc:.2f}% | Valid acc: {valid_acc:.2f}%"
        )
        list_train_acc.append(train_acc)
        list_val_acc.append(valid_acc)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

    plot_losses(list_train_acc, list_val_acc, train_losses, valid_losses)
    return model, optimizer, (train_losses, valid_losses)

## 2  Dataset, Model, and Execution

In [ ]:
class DogCatDataset(Dataset):
    """Portable image classification dataset reading train/test class subfolders."""

    def __init__(self, root_dir: str | Path, transform=None) -> None:
        self.root_dir = Path(root_dir)
        self.list_images_path: list[Path] = []
        self.list_labels: list[str] = []
        self.class_to_idx = {"dogs": 0, "cats": 1}
        self.transform = transform

        for sub_dir in sorted(self.root_dir.iterdir()):
            if not sub_dir.is_dir() or sub_dir.name not in self.class_to_idx:
                continue
            for image_path in sorted(sub_dir.iterdir()):
                if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.list_images_path.append(image_path)
                    self.list_labels.append(sub_dir.name)

    def __len__(self) -> int:
        return len(self.list_images_path)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        image = cv2.imread(str(self.list_images_path[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        label = np.array(self.class_to_idx[self.list_labels[idx]], dtype=np.int64)

        if self.transform is not None:
            res = self.transform(image=image)
            image = res["image"].astype(np.float32)
        else:
            image = image.astype(np.float32) / 255.0

        image = image.transpose(2, 0, 1)
        return image, label

In [ ]:
class Net(nn.Module):
    """Lightweight ResNet-18 classifier with a 3x3 input stem for 2-class tasks."""

    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        self.model = resnet18(weights=None, num_classes=num_classes)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


DATA_DIR = Path("../data/sample_dataset")
norm_transform = albumentations.Compose([albumentations.Normalize()])

train_data = DogCatDataset(DATA_DIR / "train", transform=norm_transform)
test_data = DogCatDataset(DATA_DIR / "test", transform=norm_transform)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

model, optimizer, _ = training_loop(
    model, criterion, optimizer, train_loader, test_loader, epochs=5, device=device
)